# 02 Feature Extraction

**Purpose.** Single full pass over every ion list, accumulating the mass profile, coarse 2-D map, and image representations.

**Thesis reference.** §4.5 Representations

**Inputs / outputs.** See the data-layout table in the repository README. Paths near the top of the notebook (e.g. `PROFILE_DIR`, `IMAGE_ROOT`, `LABELS`) are relative to the repository root and assume the folder layout described there.

---

In [4]:
DATA_ROOT = "../ahen_data/filtered_tsv_2"
IMAGE_ROOT = "images224"
PROFILE_DIR = "profiles"
FILE_PREFIX = "WCmsdata_s"

IMAGE_SIZE = 224          # pixels per axis
MZ_RESOLUTION = 0.02      # Da per fine profile bin
RT_BINS = 64              # rows in the coarse 2-D map
COARSE_MZ_BINS = 512      # columns in the coarse 2-D map

CHUNK_ROWS = 4_000_000
SKIPROWS = 1              # header row confirmed present by 00_inspect
LIMIT_FILES = None

In [5]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

MZ_MIN, MZ_MAX = 50.0, 1650.0
RT_MIN, RT_MAX = 0.0, 37.0
MZ_BANDS = [(50.0, 550.0), (550.0, 1100.0), (1100.0, 1650.0)]
ISOTOPE_SPACING = 1.0033548

REPRESENTATIONS = [
    "grayscale_heatmap", "pseudocolor_heatmap", "rgb_mz_split",
    "scatterplot", "tic_plot", "lineplot_intensity", "recurrence_plot",
]

# Fast binning
def bin_index(vals, lo, hi, n):
    """Map values to bin indices in [0, n-1]; -1 for out of range."""
    idx = ((vals - lo) * (n / (hi - lo))).astype(np.int32)
    idx[(vals < lo) | (vals >= hi)] = -1
    return idx

def accumulate2d(acc, row_idx, col_idx, weights, n_rows, n_cols):
    """Add weights into a flat 2-D accumulator using bincount."""
    ok = (row_idx >= 0) & (col_idx >= 0)
    if not ok.any():
        return 0
    flat = row_idx[ok].astype(np.int64) * n_cols + col_idx[ok]
    acc += np.bincount(flat, weights=weights[ok],
                       minlength=n_rows * n_cols).reshape(n_rows, n_cols)
    return int((~ok).sum())

# Rendering
def log_normalise(a):
    a = np.log1p(a)
    lo, hi = a.min(), a.max()
    return (a - lo) / (hi - lo) if hi > lo else np.zeros_like(a)

def to_u8(a):
    return (np.clip(a, 0.0, 1.0) * 255.0).astype(np.uint8)

def jet(x):
    x = np.clip(x, 0.0, 1.0)
    r = np.clip(1.5 - np.abs(4.0 * x - 3.0), 0.0, 1.0)
    g = np.clip(1.5 - np.abs(4.0 * x - 2.0), 0.0, 1.0)
    b = np.clip(1.5 - np.abs(4.0 * x - 1.0), 0.0, 1.0)
    return np.stack([r, g, b], axis=-1)

def curve_to_image(y, size):
    img = np.zeros((size, size), dtype=np.uint8)
    if len(y) != size:
        y = np.interp(np.linspace(0, len(y) - 1, size), np.arange(len(y)), y)
    lo, hi = y.min(), y.max()
    yn = (y - lo) / (hi - lo) if hi > lo else np.zeros_like(y)
    rows = ((1.0 - yn) * (size - 1)).astype(int)
    for x in range(size - 1):
        r0, r1 = rows[x], rows[x + 1]
        img[min(r0, r1): max(r0, r1) + 1, x] = 255
    img[rows[-1], size - 1] = 255
    return img

def recurrence_from_series(y, size, eps_frac=0.1):
    if len(y) != size:
        y = np.interp(np.linspace(0, len(y) - 1, size), np.arange(len(y)), y)
    lo, hi = y.min(), y.max()
    yn = (y - lo) / (hi - lo + 1e-12)
    d = np.abs(yn[:, None] - yn[None, :])
    eps = eps_frac * d.max() if d.max() > 0 else 1e-12
    return ((d < eps).astype(np.uint8)) * 255

# One file, one pass
def process(filepath, size, n_mz_fine):
    full = np.zeros((size, size), dtype=np.float64)
    bands = [np.zeros((size, size), dtype=np.float64) for _ in MZ_BANDS]
    coarse = np.zeros((RT_BINS, COARSE_MZ_BINS), dtype=np.float64)
    fine = np.zeros(n_mz_fine, dtype=np.float64)
    tic = np.zeros(size, dtype=np.float64)

    n_rows = 0
    n_clipped = 0

    reader = pd.read_csv(
        filepath, sep="\t", skiprows=SKIPROWS,
        names=["time", "mz", "intensity"],
        dtype=np.float32, chunksize=CHUNK_ROWS,
    )

    for ch in reader:
        rt = ch["time"].to_numpy()
        mz = ch["mz"].to_numpy()
        it = ch["intensity"].to_numpy(dtype=np.float64)

        good = np.isfinite(rt) & np.isfinite(mz) & np.isfinite(it)
        if not good.all():
            rt, mz, it = rt[good], mz[good], it[good]
        n_rows += len(rt)

        i_mz = bin_index(mz, MZ_MIN, MZ_MAX, size)
        i_rt = bin_index(rt, RT_MIN, RT_MAX, size)
        n_clipped += accumulate2d(full, i_mz, i_rt, it, size, size)

        for k, (lo, hi) in enumerate(MZ_BANDS):
            m = (mz >= lo) & (mz < hi)
            if m.any():
                accumulate2d(bands[k], bin_index(mz[m], lo, hi, size),
                             i_rt[m], it[m], size, size)

        accumulate2d(coarse,
                     bin_index(rt, RT_MIN, RT_MAX, RT_BINS),
                     bin_index(mz, MZ_MIN, MZ_MAX, COARSE_MZ_BINS),
                     it, RT_BINS, COARSE_MZ_BINS)

        i_fine = bin_index(mz, MZ_MIN, MZ_MAX, n_mz_fine)
        ok = i_fine >= 0
        fine += np.bincount(i_fine[ok], weights=it[ok], minlength=n_mz_fine)

        ok_rt = i_rt >= 0
        tic += np.bincount(i_rt[ok_rt], weights=it[ok_rt], minlength=size)

    return full, bands, coarse, fine, tic, n_rows, n_clipped

def write_images(pid, full, bands, tic, size, image_root):
    t = {r: Path(image_root) / r / f"{pid}.png" for r in REPRESENTATIONS}
    gray = log_normalise(full)

    Image.fromarray(to_u8(gray), mode="L").save(t["grayscale_heatmap"])
    Image.fromarray(to_u8(jet(gray)), mode="RGB").save(
        t["pseudocolor_heatmap"])
    rgb = np.stack([to_u8(log_normalise(b)) for b in bands], axis=-1)
    Image.fromarray(rgb, mode="RGB").save(t["rgb_mz_split"])

    occ = np.zeros_like(gray)
    occ[full > 0] = gray[full > 0]
    Image.fromarray(to_u8(occ), mode="L").save(t["scatterplot"])

    Image.fromarray(curve_to_image(tic, size), mode="L").save(t["tic_plot"])
    Image.fromarray(curve_to_image(full.max(axis=0), size), mode="L").save(
        t["lineplot_intensity"])
    Image.fromarray(recurrence_from_series(tic, size), mode="L").save(
        t["recurrence_plot"])

def main():
    root = Path(DATA_ROOT)
    files = sorted(root.glob("*.tsv"))
    if LIMIT_FILES:
        files = files[:LIMIT_FILES]
    if not files:
        raise SystemExit(f"No .tsv files in {root.resolve()}")

    for r in REPRESENTATIONS:
        (Path(IMAGE_ROOT) / r).mkdir(parents=True, exist_ok=True)
    Path(PROFILE_DIR).mkdir(parents=True, exist_ok=True)

    n_mz_fine = int(round((MZ_MAX - MZ_MIN) / MZ_RESOLUTION))
    mz_centres = (MZ_MIN + (np.arange(n_mz_fine) + 0.5) * MZ_RESOLUTION)

    print(f"Files: {len(files)}")
    print(f"Image grid: {IMAGE_SIZE}x{IMAGE_SIZE}  -> "
          f"{(MZ_MAX - MZ_MIN) / IMAGE_SIZE:.3f} Da/px, "
          f"{(RT_MAX - RT_MIN) / IMAGE_SIZE:.4f} min/px")
    print(f"Fine profile: {n_mz_fine:,} bins at {MZ_RESOLUTION} Da "
          f"({ISOTOPE_SPACING / MZ_RESOLUTION:.0f} bins per isotope step)\n")

    pids, fines, coarses, stats = [], [], [], {}
    t_start = time.time()

    for i, f in enumerate(files, 1):
        pid = f.stem.replace(FILE_PREFIX, "")
        t0 = time.time()
        full, bands, coarse, fine, tic, n_rows, n_clip = process(
            f, IMAGE_SIZE, n_mz_fine)

        if n_rows == 0:
            print(f"[{i}/{len(files)}] {pid}  EMPTY, skipped")
            continue

        write_images(pid, full, bands, tic, IMAGE_SIZE, IMAGE_ROOT)
        pids.append(pid)
        fines.append(fine.astype(np.float32))
        coarses.append(coarse.astype(np.float32))
        stats[pid] = {"n_rows": n_rows, "n_clipped": n_clip,
                      "total_intensity": float(fine.sum())}

        dt = time.time() - t0
        elapsed = time.time() - t_start
        eta = elapsed / i * (len(files) - i)
        print(f"[{i}/{len(files)}] {pid}  {n_rows:>12,} rows  "
              f"{dt:5.1f}s  clipped={n_clip:,}  "
              f"ETA {eta / 60:.0f} min")

    F = np.stack(fines)
    C = np.stack(coarses)
    np.save(Path(PROFILE_DIR) / "mz_profiles.npy", F)
    np.save(Path(PROFILE_DIR) / "coarse_maps.npy", C)
    np.save(Path(PROFILE_DIR) / "mz_centres.npy", mz_centres)
    with open(Path(PROFILE_DIR) / "profile_pids.json", "w") as fh:
        json.dump(pids, fh, indent=1)

    meta = {
        "n_patients": len(pids),
        "image_size": IMAGE_SIZE,
        "mz_range": [MZ_MIN, MZ_MAX],
        "rt_range": [RT_MIN, RT_MAX],
        "mz_per_pixel": (MZ_MAX - MZ_MIN) / IMAGE_SIZE,
        "rt_per_pixel": (RT_MAX - RT_MIN) / IMAGE_SIZE,
        "mz_resolution_da": MZ_RESOLUTION,
        "n_mz_bins": n_mz_fine,
        "rt_bins": RT_BINS,
        "coarse_mz_bins": COARSE_MZ_BINS,
        "isotope_spacing_da": ISOTOPE_SPACING,
        "downsampling": "none; all ions used",
        "axis_convention": "image row -> m/z ascending, column -> RT ascending",
        "per_file": stats,
    }
    for d in (IMAGE_ROOT, PROFILE_DIR):
        with open(Path(d) / "generation_metadata.json", "w") as fh:
            json.dump(meta, fh, indent=2)

    total_rows = sum(s["n_rows"] for s in stats.values())
    total_clip = sum(s["n_clipped"] for s in stats.values())
    print(f"\nPatients processed: {len(pids)}")
    print(f"Total ions binned:  {total_rows:,}")
    print(f"Outside m/z or RT range: {total_clip:,} "
          f"({total_clip / max(total_rows, 1):.4%})")
    print(f"Profiles: {F.shape}  ({F.nbytes / 1e6:.0f} MB)")
    print(f"Elapsed: {(time.time() - t_start) / 60:.1f} min")

In [6]:
main()

Files: 597
Image grid: 224x224  -> 7.143 Da/px, 0.1652 min/px
Fine profile: 80,000 bins at 0.02 Da (50 bins per isotope step)

[1/597] P0001    28,132,601 rows    7.0s  clipped=5,738  ETA 69 min
[2/597] P0002    28,153,190 rows    6.3s  clipped=6,835  ETA 66 min
[3/597] P0004    28,470,787 rows    6.4s  clipped=5,767  ETA 65 min
[4/597] P0005    28,126,931 rows    6.2s  clipped=5,681  ETA 64 min
[5/597] P0006    28,565,262 rows    6.2s  clipped=5,968  ETA 63 min
[6/597] P0007    27,640,887 rows   17.7s  clipped=5,664  ETA 82 min
[7/597] P0009    28,348,645 rows   20.6s  clipped=5,827  ETA 99 min
[8/597] P0010    28,233,067 rows   22.2s  clipped=6,349  ETA 114 min
[9/597] P0011    28,299,205 rows   22.8s  clipped=5,719  ETA 126 min
[10/597] P0012    28,562,099 rows   21.1s  clipped=5,924  ETA 134 min
[11/597] P0013    28,407,704 rows   22.1s  clipped=5,766  ETA 141 min
[12/597] P0014    28,183,289 rows   21.1s  clipped=5,707  ETA 146 min
[13/597] P0015    28,088,845 rows   20.3s  clippe